In [12]:
!pip uninstall -y torch torchtext torchvision torchaudio && pip install torch==2.3.0 torchvision==0.18.0 torchaudio==2.3.0 --index-url https://download.pytorch.org/whl/cu121
!pip install torchtext --no-cache-dir
!pip install einops -q


Found existing installation: torch 2.9.0+cpu
Uninstalling torch-2.9.0+cpu:
  Successfully uninstalled torch-2.9.0+cpu
Found existing installation: torchvision 0.24.0+cpu
Uninstalling torchvision-0.24.0+cpu:
  Successfully uninstalled torchvision-0.24.0+cpu
Found existing installation: torchaudio 2.9.0+cpu
Uninstalling torchaudio-2.9.0+cpu:
  Successfully uninstalled torchaudio-2.9.0+cpu
Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.9/780.9 MB 765.4 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 44.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 63.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 76.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 25.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 97.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 613.1 kB/s 

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
data_path = "/content/drive/MyDrive/Group Project 2025-2026/im2latex_resized/im2latex_resized64"

In [13]:
import torch
import torchvision
import numpy as np
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os

In [ ]:
class ImageDataset(Dataset):
  def __init__(self,dir, transform = None):
    self.dir = dir
    self.image_files = [
          f for f in os.listdir(dir) if f.endswith('.png')
      ]
    self.transform = transform
  def __len__(self):
    return len(self.image_files)
  def __getitem__(self, index):
    img_path = os.path.join(self.dir, self.image_files[index])
    image = Image.open(img_path).convert("L")
    if self.transform:
      image = self.transform(image)
    return image

In [ ]:
transform = torchvision.transforms.Compose([
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(
        mean=[0.5],
        std=[0.5]
    )
])
dataset = ImageDataset(dir=data_path,transform=transform)

# Train test split
n = int(len(dataset)*0.8)
train_dataset = torch.utils.data.Subset(dataset, range(0, n))
test_dataset  = torch.utils.data.Subset(dataset, range(n, len(dataset)))
print(len(train_dataset),len(test_dataset),len(dataset))

# Train, test loader
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)

OSError: [Errno 5] Input/output error: '/content/drive/MyDrive/Group Project 2025-2026/im2latex_resized/im2latex_resized64'

In [ ]:
class CnnEncoder(nn.Module):
    def __init__(self,input_channel, height, width, embedding_dim):
        super().__init__()
        self.features = nn.Sequential(
            #Block 1
            nn.Conv2d(input_channel,32,kernel_size=3,stride=1,padding=1),
            nn.Conv2d(32,32,kernel_size=3,stride=1,padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            #Block 2
            nn.Conv2d(32,64,kernel_size=3,stride=1,padding=1),
            nn.Conv2d(64,64,kernel_size=3,stride=1,padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            #Block 3
            nn.Conv2d(64,128,kernel_size=3,stride=1,padding=1),
            nn.Conv2d(128,128,kernel_size=3,stride=1,padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            #Block 4
            nn.Conv2d(128,256,kernel_size=3,stride=1,padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256,256,kernel_size=3,stride=1,padding=1),
            nn.ReLU(inplace=True),

            #Block 5
            nn.Conv2d(256,512,kernel_size=3,stride=1,padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512,512,kernel_size=3,stride=1,padding=1),
            nn.ReLU(inplace=True)

        )
        with torch.no_grad():
            test = torch.randn(1,input_channel,height,width)
            output = self.features(test)
            flatten_size = output.view(output.shape[0], -1).shape[1]
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flatten_size,embedding_dim),
            nn.ReLU()
        )
    def forward (self,x):
        out = self.features(x)
        out = self.fc(out)
        return out

In [26]:
img_test =  torch.randint(0, 2, (4,1 ,50, 200))
img_test = img_test.float()

In [27]:

embedding_dim = 256
encoder = CnnEncoder(input_channel=1,height=50,width=200, embedding_dim=embedding_dim)

output = encoder.forward(img_test)

print(output)
print(output.shape)


NameError: name 'CnnEncoder' is not defined

In [23]:
import torchvision.models as models
class ResnetEncoder(nn.Module):
    def __init__(self, embed_size, fine_tune = True):
        super().__init__()
        weight = models.ResNet50_Weights.DEFAULT
        self.resnet50 = models.resnet50(weight)

        for param in self.resnet50.parameters():
            param.requires_grad = fine_tune
        self.resnet50.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)

        in_features = self.resnet50.fc.in_features
        modules = list(self.resnet50.children())[:-1]
        self.resnet50 = nn.Sequential(*modules)

        self.fc = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, embed_size)
        )

    def forward(self, images):

        features = self.resnet50(images) #B , C , H, W
        features = features.view(features.size(0), -1)
        features = self.fc(features)
        return features

In [29]:
resnet = ResnetEncoder(embed_size=256)
output = resnet.forward(img_test)
print(output)
print(output.shape)

tensor([[-0.7855,  0.0367,  0.2626,  ...,  0.1236, -0.9365,  0.2433],
        [-0.5512,  0.1093, -0.8578,  ..., -0.6551, -0.8545, -0.2942],
        [-0.4966, -0.7335, -0.6098,  ...,  0.6052,  0.9231,  0.5243],
        [-0.0300, -0.5088, -0.0667,  ...,  0.1402, -0.2144,  0.4174]],
       grad_fn=<AddmmBackward0>)
torch.Size([4, 256])


In [20]:
renet = models.resnet50()
print(list(renet.children()))



[Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False), BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True), ReLU(inplace=True), MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False), Sequential(
  (0): Bottleneck(
    (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (downsample): Sequential(
      (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (1): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=Tr